# Fan Shroud ML Framework

This notebook turns the fan-shroud test table into a validated model that predicts cooler air flows and fan power for untested shroud geometries, with an uncertainty, and recommends which runs to measure next.

**How to read it.** Text cells explain what each step does and why; code cells are short and call functions from `fanml.py`. Each section ends with a *Takeaway for engineers*: a general explanation plus findings printed for the dataset you selected. Everything specific to a sheet (columns, units, formulas, open assumptions) lives in `settings.py`. For plain-language background on the methods, open `guide/fan_ml_methods_guide.html` in a browser.

| Section | What it does | Status |
| --- | --- | --- |
| 0 | Settings and open assumptions | ready |
| 1 | Load and validate the data | ready |
| 2 | Exploratory analysis | ready |
| 3 | Structure (unsupervised) | ready |
| 4 | Feature engineering | ready |
| 5 | Model zoo | ready |
| 6 | Validation and selection | ready |
| 7 | Interpretation | ready |
| 8 | Scenario prediction | ready |
| 9 | Optimisation and next experiments | ready |
| 10 | Conclusions and data recommendations | milestone M6 |

## 0. Settings

Choose the dataset and, for the plastic shroud, how to treat Run 17 (its total does not match the sum of its cooler flows):

- `DATASET`: `"steel"` (showcase: 29 runs, flows and fan power), `"plastic"` (22 runs, flows), `"ldr700"` (7 runs, one parameter; a 1-D appendix).
- `VARIANT`: `"recorded"` keeps the workbook values (default). For the plastic shroud, `"corrected"` replaces Run 17's total by its component sum and `"excluded"` drops Run 17.

In [ ]:
DATASET = "steel"     # "steel", "plastic" or "ldr700"
VARIANT = "recorded"  # "recorded"; plastic also "corrected" or "excluded"

In [ ]:
%matplotlib inline
import pandas as pd
from IPython.display import display

import fanml as fm

fm.set_style()
pd.set_option("display.max_colwidth", 140)
cfg = fm.get_config(DATASET)
print(f"Dataset: {cfg['title']} (sheet '{cfg['sheet']}'), variant: {VARIANT}")
fm.environment_info().to_frame().T

**Open assumptions.** The engineers who produced the data have not answered our questions yet, so the framework runs on documented defaults. Each default is listed below with its id; the ids reappear in the data-health report and in figure legends. When an answer arrives, change the matching entry in `settings.py`, set its status to `confirmed` and re-run.

In [ ]:
fm.assumptions_table(cfg)

## 1. Load and validate

The sheet is read by **column name**, never by position, so a reordered column cannot be mistaken for another one (the plastic sheet orders its depths P1, P3, P2). The raw workbook is never edited: corrections are variants defined in `settings.py`.

Before any modelling, `validate()` runs eight checks:

| Check | Why |
| --- | --- |
| Missing values | A run without inputs cannot be modelled. |
| Components add up to the total | Cooler flows must sum to the total flow; a mismatch points to a typo or a separately measured total. |
| Sheet formulas recalculate | Typed-in copies of formula columns (immersion) must match the formula. |
| Repeated values at range edges | Several runs at exactly the same limit suggest an optimiser chose them. |
| Duplicate designs | Two runs with identical inputs would reveal the measurement noise. |
| Number precision | Mixed precision can mean mixed sources (simulation and measurement). |
| Correlated inputs | Inputs that change together cannot be separated by any model. |
| Open assumptions | What still waits for the engineers. |

In [ ]:
df = fm.load_dataset(cfg, variant=VARIANT)
print(f"{len(df)} rows, {len(fm.design_rows(df, cfg))} with a complete set of inputs")
df.head()

In [ ]:
report = fm.validate(df, cfg)
report.save(cfg)          # outputs/<dataset>/data_health.md and tables/data_health.csv
report.checks

In [ ]:
if "closure" in report.details:
    print("Largest differences between the total and the sum of the cooler flows:")
    display(report.details["closure"].head(6).round(4))

### Takeaway for engineers

- **FLAG** means the finding changes numbers downstream; **NOTE** means it is worth knowing; **OK** means nothing was found.
- Flagged runs stay in the data under the default variant and are marked with a red ring in the plots. Re-running with `VARIANT = "corrected"` or `"excluded"` shows how much a flag matters.
- The report is saved to `outputs/<dataset>/data_health.md` for the test engineers.

In [ ]:
fm.findings_validation(report)

## 2. Exploratory analysis

First we add the physics coordinates (immersion, split), the flow fractions and the outputs relative to baseline. Their formulas are in `settings.py`; section 4 explains why we use them. Then we look at the data four ways: summary statistics, outputs against immersion, every feature against the key outputs, and three measures of association.

In [ ]:
data = fm.add_features(df, cfg)
fm.summary_table(data, cfg)

### 2.1 Outputs against the key variable

Each panel shows one output against immersion (Parameter 1 for LDR 700). Small numbers are run numbers, a black ring marks the baseline, a red ring a flagged run. The last panel shows the total flow divided by the baseline value, which makes datasets comparable.

In [ ]:
fig = fm.plot_responses(data, cfg, flagged=report.flagged_runs)
fm.save_fig(fig, cfg, "02_1_responses");

### 2.2 Every feature against the key outputs

One row per feature, one column per key output (total flow, charge-air cooler flow and fan power where available). A clear pattern in a row means that feature matters; a shapeless cloud means it does not, at least on its own.

In [ ]:
fig = fm.plot_input_output_grid(data, cfg)
fm.save_fig(fig, cfg, "02_2_feature_output_grid");

### 2.3 How strongly features and outputs are linked

Three measures, each answering a slightly different question:

- **Pearson r**: how well a straight line fits (−1 to 1).
- **Spearman ρ**: whether the output steadily rises or falls with the feature, in any shape (−1 to 1).
- **Mutual information** (MI): any dependence at all, including a rise-then-fall peak (0 = none). With about 25 runs, use MI to rank features, not as a precise number.

In [ ]:
corr = fm.correlation_table(data, cfg)
fm.save_table(corr, cfg, "02_3_correlations")
fig = fm.plot_correlations(corr, cfg)
fm.save_fig(fig, cfg, "02_3_correlations");

### 2.4 Coverage: where the tests are

Models are trustworthy only where there is data. The panels show the tested designs in the physics coordinates; the last panel shows the tested values of the key variable with the largest empty interval shaded. Predictions inside a gap are guesses until a run is measured there.

In [ ]:
fig = fm.plot_coverage(data, cfg)
fm.save_fig(fig, cfg, "02_4_coverage")
fm.find_gaps(fm.design_rows(data, cfg)[cfg["key_x"]]).round(3)

### Takeaway for engineers

- Look at section 2.1 first: it shows the main pattern and which runs stand out.
- A feature with a strong MI but a weak Spearman ρ has a peak or a valley: changing it helps only up to a point.
- The largest gap in coverage is where the next measurement teaches the model most.

In [ ]:
fm.findings_exploration(data, cfg, corr)

## 3. Structure (unsupervised)

Unsupervised methods look for structure without a target to predict. Three questions:

1. **How many independent patterns are in the outputs?** Principal component analysis (PCA) of the standardised outputs. One dominant component means the outputs move together ("more air through everything") and one model is nearly enough; several components mean the split between coolers changes on its own. The total flow is left out because it is the sum of the cooler flows.
2. **How independently were the inputs varied?** Correlations between inputs. Inputs that change together cannot be separated by any model; this is a property of the test plan.
3. **How is the air split between coolers?** Each cooler's share of the summed flow. A design can raise total flow and still starve one cooler.

In [ ]:
pca_res = fm.pca_outputs(data, cfg)
if pca_res is None:
    print("Only one output in this sheet: no output structure to analyse.")
else:
    fig = fm.plot_pca_outputs(pca_res, cfg)
    fm.save_fig(fig, cfg, "03_1_pca_outputs")
    display(pca_res["loadings"].round(2))

In [ ]:
inp = fm.input_structure(data, cfg)
if len(cfg["inputs"]) > 1:
    fig = fm.plot_input_structure(inp, cfg)
    fm.save_fig(fig, cfg, "03_2_input_correlation")
else:
    print("Only one input in this sheet.")

In [ ]:
if cfg["components"]:
    fig = fm.plot_fractions(data, cfg)
    fm.save_fig(fig, cfg, "03_3_flow_fractions")
    display(fm.fraction_ranges(data, cfg))
else:
    print("No cooler flows in this sheet.")

### Takeaway for engineers

- The share of variation in the first principal component tells how many separate things the model has to learn.
- A strong correlation between raw inputs comes from how the runs were chosen; the physics features (section 4) reduce it.
- The cooler whose share varies most is the one to watch in optimisation: it becomes a constraint in section 9.

In [ ]:
fm.findings_structure(data, cfg, pca_res, inp)

## 4. Feature engineering

With few runs, the variables a model sees matter more than the model itself. We describe the shroud the way the physics sees it:

- **Immersion**: how far the shroud ring covers the fan (share of the fan depth). It is the sum of the segment depths, rescaled.
- **Split** (ring − transition): how that cover is built. Same immersion, different split, different flow.
- **Flow fractions**: each cooler's share, so predicted cooler flows always add up to the predicted total.
- **Outputs relative to baseline**: comparable across shrouds.

Because immersion is a sum of the raw depths, it cannot sit next to all of them in one model: the features would be an exact combination of each other. The table below shows this with the condition number of the standardised feature matrix (around 1–10 is healthy; above 10¹⁰ means one column is a combination of others).

In [ ]:
check = fm.collinearity_check(data, cfg)
check

In [ ]:
FEATURE_SETS = fm.feature_sets(cfg)    # compared in section 6
derived = [c for c in data.columns if c in cfg["derived"] or c.startswith("frac_") or c.endswith("_rel")]
print("Feature sets:", FEATURE_SETS)
data[derived].head()

### Takeaway for engineers

- The physics features carry the same information as the raw depths, arranged the way the flow responds to them.
- Section 6 compares models trained on raw inputs and on physics features on runs they have not seen; the first analysis showed lower errors with physics features.

In [ ]:
fm.findings_features(check, cfg)

## 5. Model zoo

Four models, from simple to flexible. With 7–29 runs, flexibility has to earn its place: a model is only preferred if it predicts runs it has not seen better (section 6).

Neural networks are left out on purpose: with fewer than 30 runs they fit noise, and they give no uncertainty. They become worth trying with hundreds of CFD runs.

In [ ]:
fm.describe_models()

The outputs modelled for this dataset and the two feature sets compared (raw sheet inputs vs physics features from section 4) are set in `settings.py`:

In [ ]:
print("Targets:", [fm.short(cfg, t) for t in cfg["targets"]])
fm.feature_sets_for(cfg)

## 6. Validation and selection

Every score in this section comes from runs the model did **not** see while it was trained:

| Step | Question |
| --- | --- |
| 6.1 Leave-one-out | How well does each model predict a run it has not seen? Each run is predicted by a model trained on all the others. |
| 6.2 Selection | Which model do we use? The simplest candidate whose error is within 5 % of the lowest (random forest is a contrast, never chosen). |
| 6.3 Nested check | Does picking the winner flatter its score? The whole "compare, pick, predict" procedure is repeated with each run left out. |
| 6.4 Later runs | Train on the earlier runs, predict the later ones, as the model will be used in practice. |
| 6.5 Parity and errors | Where does the model miss, and is there a pattern? |
| 6.6 Uncertainty | Are the Gaussian process's bands honest? |
| 6.7 Data variants | How much does a flagged run (Run 17) change the result? |
| 6.8 Baseline check | Can the model predict a run never used in training? |

**Three error measures**, all computed on unseen runs:

| Measure | Definition | Read it as | Use it to |
| --- | --- | --- | --- |
| **Q²** | 1 − Σ(measured − predicted)² / Σ(measured − mean)², with each prediction made by a model that never saw that run | 1 = perfect, 0 = no better than always predicting the average, below 0 = worse | compare models on the same output |
| **RMSE** | √(mean of squared errors), in the output's unit and as % of the baseline value | the typical error | judge whether a model is good enough (against tolerance and measurement repeatability) |
| **Max error** | the largest single miss, % of baseline | the worst case | see whether one run dominates the error |

Q² depends on how much an output varies: Q² ≈ 1 − (RMSE / SD)², where SD is the spread of the measured values. An output that hardly changes can have a small RMSE and still a low Q². So compare models with Q², and decide whether a model is useful with RMSE and max error. The section takes two to three minutes on a laptop.

### 6.1 Leave-one-out comparison

All outputs are modelled; the key outputs (`key_targets` in the settings) are also compared on raw inputs. The figure and table show the key outputs; the full table for every output is saved to `outputs/<dataset>/tables/06_1_loo_metrics.csv`.

In [ ]:
cv = fm.cross_validate(data, cfg)
fm.save_table(cv.metrics, cfg, "06_1_loo_metrics")
fig = fm.plot_model_comparison(cv, cfg)
fm.save_fig(fig, cfg, "06_1_model_comparison")
key = cfg["key_targets"]
(cv.metrics[cv.metrics["target"].isin(key)]
   .set_index(["target", "features", "model"])[["q2", "rmse", "rmse_pct", "max_abs_err_pct"]]
   .round(3))

### 6.2 Selection

The rule prefers the simpler model when the difference is small (Occam's razor): a simpler model is easier to explain and extrapolates more calmly. It is applied to the physics features.

In [ ]:
selection = fm.select_models(cv, cfg)
fm.save_table(selection, cfg, "06_2_selection")
selection.round(3)

### 6.3 Nested check of the selection

If the nested score is close to the leave-one-out score of the chosen model, the choice did not flatter it. The last column shows how often each model won inside the loop: a stable winner is a good sign.

In [ ]:
nested = fm.nested_cv(data, cfg)
fm.save_table(nested, cfg, "06_3_nested")
nested.round(3)

### 6.4 Later-runs test

The run numbers show the order of the campaign; the later runs explored new designs. The split point is set in `settings.py` (`later_runs_from`).

In [ ]:
later = fm.later_runs_test(data, cfg)
if later is None:
    print("No later-runs split defined for this dataset.")
else:
    fm.save_table(later, cfg, "06_4_later_runs")
    print(f"Trained on {later.attrs['train_runs']} ({later.attrs['n_train']} runs), "
          f"tested on {later.attrs['test_runs']} ({later.attrs['n_test']} runs).")
    display(later.set_index(["target", "features", "model"])[["rmse", "rmse_pct", "max_abs_err_pct", "worst_run"]]
            .round(3))

### 6.5 Measured vs predicted, and where the errors are

Left: every model's unseen-run predictions against the measured values; points on the dashed line are perfect, grey bars are the Gaussian process's 95 % bands, a red ring marks a flagged run. Right: the chosen model's errors against each feature. Errors scattered evenly around zero are good; a trend means the model misses some physics.

In [ ]:
for t in cfg["key_targets"]:
    fig = fm.plot_parity(cv, cfg, t, flagged=report.flagged_runs)
    fm.save_fig(fig, cfg, f"06_5_parity_{t}")
    fig = fm.plot_residuals(cv, data, cfg, t, selection.loc[t, "model"])
    fm.save_fig(fig, cfg, f"06_5_residuals_{t}")

### 6.6 Uncertainty check

A 95 % band should contain about 95 % of the unseen runs. The **calibration** factor is the root-mean-square of the standardised errors: about 1 means honest bands, 1.6 means the bands must be widened by 1.6× before they are used for decisions. Section 8 therefore builds its intervals from the actual unseen-run errors (conformal intervals), and section 9 widens the Gaussian process's uncertainty by this factor.

In [ ]:
coverage = [fm.interval_coverage(cv, t) for t in cfg["key_targets"]]
display(pd.DataFrame(coverage).set_index("target").drop(columns="outside_95").round(2))
fig = fm.plot_gp_bands(cv, data, cfg, cfg["targets"][0])
fm.save_fig(fig, cfg, "06_6_gp_bands");

### 6.7 Data variants

For datasets with a flagged run, the chosen model is re-validated with the run as recorded, corrected and excluded. A large difference means the open question matters for the result.

In [ ]:
sens = None
if cfg["variants"]:
    t = cfg["key_targets"][0]
    sens = fm.variant_sensitivity(cfg, t, selection.loc[t, "model"])
    fm.save_table(sens, cfg, "06_7_variants")
    print(f"{fm.short(cfg, t)}, {selection.loc[t, 'model']}:")
    display(sens.round(3))
else:
    print("No data variants defined for this dataset.")

### 6.8 Baseline check

A baseline row without inputs was never used in training, so predicting it is a free independent test. Its geometry comes from the assumptions (A3); inputs that remain unknown are swept over their range.

In [ ]:
base = fm.baseline_check(data, cfg, selection)
if base is None:
    print("The baseline run has inputs and is part of the training data: no independent check here.")
else:
    fm.save_table(base, cfg, "06_8_baseline_check")
    print("Unknown inputs swept:", base.attrs["unknown_inputs"])
    display(base.round(3))

### Takeaway for engineers

- The scores estimate how well the model predicts a **new design inside the tested range**. Outside it, no score applies.
- The winning model can differ between datasets. The rule is fixed in advance; the winner is not.
- Look at all three measures: Q² to rank models, RMSE to judge usefulness, max error to spot a single bad miss. An output that barely varies (for example the fuel cooler on plastic) can have a low Q² with a small RMSE.
- Comparing raw and physics features with the same model shows what the feature engineering buys, most clearly on the later-runs test.
- Uncertainty bands are only useful if they are honest; the calibration factor says how much to widen them.
- If a data variant changes the score clearly, resolving the open question with the engineers improves the model more than any change of algorithm.

In [ ]:
fm.findings_models(cfg, cv, selection, nested, later, coverage, sens, base)

## 7. Interpretation

From here on, each output uses its chosen model from section 6, refitted on all runs. Three views:

1. **Sobol indices**: which feature drives each output over the tested ranges. *S1* is the share of the output's variation caused by a feature alone; *ST* includes its interactions with other features. Features are varied independently, so some combinations fall outside the design limits; their share is printed, and there the model extrapolates.
2. **Effect curves**: how each output changes when one feature moves away from today's design while the others stay put.
3. **Response map**: the predicted output over the two most important features.

The bands in 7.2 are 90 % prediction intervals, explained in section 8.

In [ ]:
models = fm.fit_final_models(data, cfg, selection)
conf = fm.conformal_intervals(cv, selection)
checker = fm.range_checker(data, cfg)
ref = fm.reference_design(data, cfg)
print("Today's design (reference):", ref.round(3).to_dict())

### 7.1 Which feature drives each output

In [ ]:
sobols = {t: fm.sobol_indices(models[t], data, cfg) for t in cfg["key_targets"]}
if any(v is not None for v in sobols.values()):
    fig = fm.plot_sobol(sobols, cfg)
    fm.save_fig(fig, cfg, "07_1_sobol")
    for t, v in sobols.items():
        print(f"{fm.short(cfg, t)}: {100 * v.attrs['outside_limits_share']:.0f} % of samples outside the design limits")
    relevance = fm.gp_relevance(models, cfg)
    if relevance is not None:
        print("Gaussian process length scales (short = matters, above 100 = switched off):")
        display(relevance.round(2))
else:
    relevance = None
    print("Only one feature in this dataset: no sensitivity split.")

### 7.2 Effect of each feature from today's design

Each panel moves one feature across its tested range while the others stay at today's design (dashed line). Solid line: inside the tested region; dotted: outside it. Grey dots are all tested runs; they differ in the other features, so they do not have to lie on the curve.

In [ ]:
fig = fm.plot_effect_curves(data, cfg, models, conf, ref, checker=checker)
fm.save_fig(fig, cfg, "07_2_effect_curves");

### 7.3 Response map

The predicted goal output over the two map features (`map_pair` in the settings), with the other features at today's design. Blank areas are outside the design limits. Tested runs are drawn at their own values.

In [ ]:
goal = cfg["optimisation"]["maximise"]
fig = fm.plot_response_map(data, cfg, models, conf, ref, goal, checker=checker)
if fig is None:
    print("One feature only: see the curve in section 7.2.")
else:
    fm.save_fig(fig, cfg, "07_3_response_map");

### Takeaway for engineers

- The Sobol indices rank the features by how much they move each output; a feature near zero can be set for other reasons (cost, packaging) without hurting the air flow.
- The effect curves answer "what happens if I change only this?" starting from today's design, with an honest band.
- The response map shows the best region and how steep the surroundings are: a flat region tolerates manufacturing scatter.

In [ ]:
fm.findings_interpretation(cfg, sobols, relevance)

## 8. Scenario prediction

Enter any geometry and get every modelled output with a **prediction interval** and a **range status**.

**Intervals (conformal).** They are built from the chosen model's actual errors on unseen runs (section 6), so they are as wide as the model has really been wrong, without assuming a distribution. For the Gaussian process the errors are first divided by its own uncertainty, so the band widens where the model is less sure; for the other models it has a constant width. The level is 90 % (`INTERVAL_LEVEL`): with about 25 runs, a 95 % interval would simply equal the largest error seen.

**Range status**, from least to most serious:

| Status | Meaning |
| --- | --- |
| inside tested range | trust the interval |
| far from tested runs | inside the ranges, but in a gap larger than any gap between tested runs |
| outside tested range | a physics feature (e.g. immersion) beyond the tested values: extrapolation, treat as a guess |
| outside design limits | a raw input beyond the limits in `settings.py` (A5) |

In [ ]:
display(conf.round(4))

In [ ]:
designs = fm.what_if_designs(cfg)      # examples from settings.py; add your own rows here
print("Inputs needed per design:", cfg["inputs"])
pred = fm.predict_designs(designs, data, cfg, models, conf, checker)
fm.save_table(pred, cfg, "08_what_if")
fm.show_predictions(pred, cfg)

To try your own design, add a row before running the cell, for example `designs.loc[len(designs)] = {"design": "My design", ...}` with a value for every input listed above.

**Consistency check.** Each cooler and the total are predicted by separate models. Their sum should match the predicted total; a large difference means at least one model is outside its comfort zone.

In [ ]:
if "coolers_sum_minus_total" in pred:
    display(pred[["status", "coolers_sum_minus_total"]].round(4))
else:
    print("No separate cooler models in this dataset.")

### Takeaway for engineers

- Read every prediction together with its interval and its status. Inside the tested range the interval is honest; outside it, the number is a guess, however precise it looks.
- The example designs in `settings.py` show both cases on purpose.

In [ ]:
fm.findings_prediction(cfg, data, conf, pred)

## 9. Optimisation and next experiments

**Goal and constraints** come from the settings (`optimisation`). They are a placeholder until the engineers confirm them (A6): maximise total flow while the charge-air cooler gets at least its baseline flow. The search covers a few thousand candidate designs (`POOL_SIZE`, 8,192 by default) spread evenly within the design limits (A5), keeping only those inside the tested range.

Three outputs:

1. **Optimum.** Two answers: the design with the highest predicted value, and a cautious one with the highest *lower* interval bound. The best measured run is shown for comparison.
2. **Trade-off.** Where two goals compete (flow vs fan power, or total vs charge-air cooler flow), the front of best compromises.
3. **Next runs.** A small batch to measure next:
    - *improve design*: highest expected improvement over the best measured run (Bayesian optimisation; uses a Gaussian process whose uncertainty is widened by the calibration factor from 6.6);
    - *improve model*: the largest gap inside the tested region, i.e. the design farthest from any tested run.

In [ ]:
opt = fm.optimise(data, cfg, models, conf, checker)
fm.save_table(opt, cfg, "09_1_optimum")
print("Constraints:", opt.attrs["constraints"] or "none",
      f"| feasible candidates: {opt.attrs['n_feasible']} of {opt.attrs['n_pool']}")
cols = [*cfg["inputs"], *cfg["derived"], *[c for t in cfg["key_targets"] for c in (t, f"{t}_low", f"{t}_high")]]
opt[[c for c in cols if c in opt.columns]].round(3)

In [ ]:
pool_pred, front = fm.tradeoff(data, cfg, models, conf, checker)
if front is None:
    print("No trade-off defined for this dataset.")
else:
    fig = fm.plot_tradeoff(pool_pred, front, data, cfg)
    fm.save_fig(fig, cfg, "09_2_tradeoff")

In [ ]:
calibration = fm.interval_coverage(cv, goal)["calibration"]
next_runs = fm.suggest_next_runs(data, cfg, models, conf, calibration=calibration, checker=checker)
fm.save_table(next_runs, cfg, "09_3_next_runs")
fig = fm.plot_response_map(data, cfg, models, conf, ref, goal, checker=checker, optimum=opt, proposals=next_runs)
if fig is None:
    fig = fm.plot_next_runs_1d(data, cfg, models, conf, ref, next_runs, opt)
fm.save_fig(fig, cfg, "09_3_next_runs")
next_runs[["purpose", *cfg["inputs"], *cfg["derived"], "status", "nearest_run", goal, f"{goal}_low", f"{goal}_high"]].round(3)

### Takeaway for engineers

- The optimum is a **proposal to test**, not a result: it relies on the model between tested runs and on the placeholder goal (A6) and limits (A5).
- Prefer the cautious optimum when a wrong guess is expensive.
- Measuring the proposed runs and re-running this notebook is the loop that makes the model better: each batch narrows the intervals where it matters.

In [ ]:
fm.findings_design(cfg, data, opt, front, next_runs)

## 10. Conclusions and data recommendations

Added in milestone M6, together with the handover README.